# Homework 1: Agentic RAG

## LLM Zoocamp 2026 - DataTalks

This notebook contains the code and exercises for Homework 1 on Agentic Retrieval-Augmented Generation.

## Setup

Load environment variables and import required libraries.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify that the API key is loaded
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("✓ API key loaded successfully")
else:
    print("✗ API key not found. Please check your .env file.")

## Course Material

Add and execute code from the section notes below.

In [ ]:
# TODO: Add code from section notes here

## Exercises

Complete the exercises for this homework.

In [3]:
# install missing package (in Jupyter use %pip)
%pip install openai

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\aleite\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:
uv add gitsource

Note: you may need to restart the kernel to use updated packages.


Resolved 123 packages in 205ms
Audited 119 packages in 95ms


In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

ModuleNotFoundError: No module named 'gitsource'

In [7]:
%pip install gitsource requests openai python-dotenv minsearch -q

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# Parse all files into documents
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Display the number of lesson pages
print(f"Total number of lesson pages: {len(documents)}")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\aleite\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Total number of lesson pages: 72


## Q1. How many lesson pages

In [9]:
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


## Q2. Search with minsearch

In [12]:
from minsearch import Index

# Create index with text field (content) and keyword field (filename)
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Search for the query
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, filter_dict={})

# Display results
print(f"Search query: {query}\n")
print(f"Number of results: {len(results)}")
if results:
    print(f"\nFirst result filename: {results[0]['filename']}")
    print(f"\nTop results:")
    for i, result in enumerate(results[:5], 1):
        print(f"{i}. {result['filename']}")

Search query: How does the agentic loop keep calling the model until it stops?

Number of results: 10

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md

Top results:
1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/15-frameworks.md
3. 01-agentic-rag/lessons/13-function-calling.md
4. 01-agentic-rag/lessons/11-agents-intro.md
5. 01-agentic-rag/lessons/16-other-frameworks.md


## Q3. Build RAG with adapted RAGBase

In [13]:
# Download rag_helper.py
import urllib.request

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
urllib.request.urlretrieve(url, "rag_helper.py")

print("✓ rag_helper.py downloaded successfully")

# Load the RAGBase helper
from rag_helper import RAGBase

✓ rag_helper.py downloaded successfully


In [16]:
# Create a custom RAG class for our document schema (filename + content)
class CustomRAG:
    def __init__(self, index, client, model="gpt-4o-mini"):
        self.index = index
        self.client = client
        self.model = model
    
    def search(self, query, top_k=5):
        """Search the index for relevant documents"""
        results = self.index.search(query, filter_dict={})
        return results[:top_k] if len(results) > top_k else results
    
    def build_context(self, search_results):
        """Build context from search results"""
        context_parts = []
        for result in search_results:
            context_parts.append(f"File: {result['filename']}\n{result['content']}")
        return "\n---\n".join(context_parts)
    
    def answer_query(self, query, top_k=5):
        """Answer a query using RAG"""
        # Search for relevant documents
        search_results = self.search(query, top_k)
        
        # Build context
        context = self.build_context(search_results)
        
        # Create prompt
        prompt = f"""Based on the context below, answer the following question:

Context:
{context}

Question: {query}

Answer:"""
        
        # Get response from OpenAI
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Extract metrics
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        
        return {
            "answer": answer,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "search_results": search_results
        }

# Initialize RAG with gpt-4o-mini model
rag = CustomRAG(index=index, client=openai_client, model="gpt-4o-mini")

In [19]:
# Answer the query using RAG
query = "How does the agentic loop keep calling the model until it stops?"
result = rag.answer_query(query)

# Display results
print(f"Query: {query}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"=" * 80)
print(f"Input (prompt) tokens: {result['input_tokens']}")
print(f"Output tokens: {result['output_tokens']}")
print(f"Total tokens: {result['input_tokens'] + result['output_tokens']}")
print(f"=" * 80)
print(f"\nSearch results used:")
for i, res in enumerate(result['search_results'], 1):
    print(f"{i}. {res['filename']}")

RAG QUERY SIMULATION

Query: How does the agentic loop keep calling the model until it stops?

Prompt being sent to model:
--------------------------------------------------------------------------------
Based on the context below, answer the following question:

Context:
File: 01-agentic-rag/lessons/14-agentic-loop.md
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model wi...
--------------------------------------------------------------------------------

Prompt length: 1819 characters
Estimated input (prompt) tokens: ~454 tokens
(Actual token count from OpenAI AP

In [20]:
# Execute the RAG with real OpenAI API call
print("Calling RAG with OpenAI API...\n")

query = "How does the agentic loop keep calling the model until it stops?"
result = rag.answer_query(query)

# Display results
print("=" * 80)
print("RAG RESPONSE")
print("=" * 80)
print(f"\nQuery: {query}\n")
print(f"Answer:\n{result['answer']}\n")
print("=" * 80)
print(f"\nToken Usage:")
print(f"  Input (prompt) tokens: {result['input_tokens']}")
print(f"  Output tokens: {result['output_tokens']}")
print(f"  Total tokens: {result['input_tokens'] + result['output_tokens']}")
print("=" * 80)
print(f"\nSearch results used:")
for i, res in enumerate(result['search_results'], 1):
    print(f"{i}. {res['filename']}")

Calling RAG with OpenAI API...

RAG RESPONSE

Query: How does the agentic loop keep calling the model until it stops?

Answer:
The agentic loop keeps calling the model until it stops by using a `while True` loop that continuously executes until it receives a response from the model that does not contain any function calls. Here's how it works step-by-step:

1. **Initialization**: The loop starts with an iteration counter set to 1 and a flag (`has_function_calls`) initialized to `False`.

2. **Model Response**: Within each iteration, the model is called with the current messages (which include instructions, user input, and potentially previous function call outputs).

3. **Processing Output**: The model's response is processed to check if it contains any function calls:
   - If the response includes a function call (indicated by checking the response's output type), the corresponding function is executed (e.g., a search operation).
   - The result of the function call is then added back

In [21]:
# Summary of Q3 Results
print("\n" + "=" * 80)
print("Q3 SUMMARY - RAG WITH REAL API")
print("=" * 80)
print(f"\nInput (Prompt) Tokens: {result['input_tokens']}")
print(f"Output Tokens: {result['output_tokens']}")
print(f"\n✅ The RAG successfully answered the query using gpt-4o-mini!")
print(f"\nSearch documents used: {len(result['search_results'])}")
for i, res in enumerate(result['search_results'], 1):
    print(f"  {i}. {res['filename']}")


Q3 SUMMARY - RAG WITH REAL API

Input (Prompt) Tokens: 7091
Output Tokens: 355

✅ The RAG successfully answered the query using gpt-4o-mini!

Search documents used: 5
  1. 01-agentic-rag/lessons/14-agentic-loop.md
  2. 01-agentic-rag/lessons/15-frameworks.md
  3. 01-agentic-rag/lessons/13-function-calling.md
  4. 01-agentic-rag/lessons/11-agents-intro.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md


## Q4. Chunking Documents

In [22]:
# Split documents into chunks using sliding window
from gitsource import chunk_documents

# Create chunks with size=2000 characters and step=1000
# This creates overlapping chunks for better context preservation
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Original documents: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(f"\nFirst few chunks info:")
for i, chunk in enumerate(chunks[:3]):
    content_length = len(chunk.get('content', ''))
    print(f"  Chunk {i+1}: filename={chunk['filename']}, content_length={content_length}")

Original documents: 72
Chunks created: 295

First few chunks info:
  Chunk 1: filename=01-agentic-rag/lessons/01-intro.md, content_length=2000
  Chunk 2: filename=01-agentic-rag/lessons/01-intro.md, content_length=2000
  Chunk 3: filename=01-agentic-rag/lessons/01-intro.md, content_length=1183


In [23]:
# Create a new index with chunks for better search precision
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Index all chunks
index_chunks.fit(chunks)

# Search the same query in the chunked index
query = "How does the agentic loop keep calling the model until it stops?"
results_chunks = index_chunks.search(query, filter_dict={})

print(f"\n{'='*80}")
print("COMPARISON: Original vs Chunked Index")
print(f"{'='*80}")
print(f"\nQuery: {query}\n")

# Results from original index
results_original = index.search(query, filter_dict={})
print(f"Original index (72 documents):")
print(f"  Results found: {len(results_original)}")
print(f"  First result: {results_original[0]['filename']}")

# Results from chunked index
print(f"\nChunked index (295 chunks):")
print(f"  Results found: {len(results_chunks)}")
print(f"  First result: {results_chunks[0]['filename']}")

print(f"\n{'='*80}")
print(f"Improvement: Chunking splits {len(documents)} documents into {len(chunks)} chunks")
print(f"More precise retrieval: documents are broken into smaller, focused pieces")
print(f"{'='*80}")


COMPARISON: Original vs Chunked Index

Query: How does the agentic loop keep calling the model until it stops?

Original index (72 documents):
  Results found: 10
  First result: 01-agentic-rag/lessons/14-agentic-loop.md

Chunked index (295 chunks):
  Results found: 10
  First result: 01-agentic-rag/lessons/14-agentic-loop.md

Improvement: Chunking splits 72 documents into 295 chunks
More precise retrieval: documents are broken into smaller, focused pieces


In [24]:
# Q4 Answer: How many chunks are created?
print(f"\n{'='*80}")
print("Q4 ANSWER: Chunking Results")
print(f"{'='*80}")
print(f"\nWith chunk_documents(documents, size=2000, step=1000):")
print(f"  Input: {len(documents)} documents")
print(f"  Output: {len(chunks)} chunks")
print(f"\nChunking parameters:")
print(f"  - Window size: 2000 characters")
print(f"  - Step size: 1000 characters")
print(f"  - This creates overlapping chunks for better context preservation")
print(f"  - Average chunks per document: {len(chunks) / len(documents):.2f}")
print(f"\nBenefits of chunking:")
print(f"  ✓ More precise retrieval (smaller pieces = better matches)")
print(f"  ✓ Overlapping windows preserve context across chunk boundaries")
print(f"  ✓ Reduces irrelevant content in retrieved results")
print(f"{'='*80}")


Q4 ANSWER: Chunking Results

With chunk_documents(documents, size=2000, step=1000):
  Input: 72 documents
  Output: 295 chunks

Chunking parameters:
  - Window size: 2000 characters
  - Step size: 1000 characters
  - This creates overlapping chunks for better context preservation
  - Average chunks per document: 4.10

Benefits of chunking:
  ✓ More precise retrieval (smaller pieces = better matches)
  ✓ Overlapping windows preserve context across chunk boundaries
  ✓ Reduces irrelevant content in retrieved results


In [25]:
# Detailed breakdown of chunk structure
print(f"\n{'='*80}")
print("Q4: How many chunks are created?")
print(f"{'='*80}")
print(f"\nANSWER: {len(chunks)} chunks")
print(f"\nParameters:")
print(f"  size = 2000 (window size in characters)")
print(f"  step = 1000 (sliding window step in characters)")
print(f"  overlap = size - step = 2000 - 1000 = 1000 characters")
print(f"\nChunk structure (each chunk contains):")
print(f"  ✓ filename (original field)")
print(f"  ✓ content (chunk text of up to 2000 characters)")
print(f"  ✓ start (offset in the original document)")
print(f"\nExamples of chunks from the first document:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n  Chunk {i+1}:")
    print(f"    - filename: {chunk['filename']}")
    print(f"    - start: {chunk.get('start', 'N/A')}")
    print(f"    - content length: {len(chunk.get('content', ''))} chars")
    print(f"    - preview: {chunk['content'][:100]}...")

print(f"\n{'='*80}")
print(f"FINAL ANSWER FOR Q4: {len(chunks)} chunks")
print(f"{'='*80}")


Q4: How many chunks are created?

ANSWER: 295 chunks

Parameters:
  size = 2000 (window size in characters)
  step = 1000 (sliding window step in characters)
  overlap = size - step = 2000 - 1000 = 1000 characters

Chunk structure (each chunk contains):
  ✓ filename (original field)
  ✓ content (chunk text of up to 2000 characters)
  ✓ start (offset in the original document)

Examples of chunks from the first document:

  Chunk 1:
    - filename: 01-agentic-rag/lessons/01-intro.md
    - start: 0
    - content length: 2000 chars
    - preview: # Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxU...

  Chunk 2:
    - filename: 01-agentic-rag/lessons/01-intro.md
    - start: 1000
    - content length: 2000 chars
    - preview: the next
word based on what you typed so far.

A large language model does the same thing, but at a ...

  Chunk 3:
    - filename: 01-agentic-rag/lessons/01-intro.md
    - start: 2000
    - content length: 1183 chars

## Q5. RAG with Chunking - Token Comparison

In [26]:
# Create a RAG instance using the chunked index
rag_chunks = CustomRAG(index=index_chunks, client=openai_client, model="gpt-4o-mini")

# Answer the same query using chunks
query = "How does the agentic loop keep calling the model until it stops?"
result_chunks = rag_chunks.answer_query(query)

# Display results
print(f"{'='*80}")
print("Q5: RAG WITH CHUNKING")
print(f"{'='*80}")
print(f"\nQuery: {query}\n")
print(f"Answer (from chunked index):\n{result_chunks['answer'][:500]}...\n")
print(f"{'='*80}")
print("TOKEN COMPARISON")
print(f"{'='*80}")
print(f"\nQ3 (Full documents - 72 docs):")
print(f"  Input tokens: 7091")
print(f"\nQ5 (Chunked index - 295 chunks):")
print(f"  Input tokens: {result_chunks['input_tokens']}")
print(f"\nDifference:")
reduction = 7091 - result_chunks['input_tokens']
percentage = (reduction / 7091) * 100
print(f"  Tokens saved: {reduction}")
print(f"  Percentage: {percentage:.1f}% reduction")
print(f"\nRatio:")
ratio = 7091 / result_chunks['input_tokens']
if ratio >= 25:
    print(f"  ~30× fewer tokens")
elif ratio >= 8:
    print(f"  ~10× fewer tokens")
elif ratio >= 2.5:
    print(f"  ~3× fewer tokens")
else:
    print(f"  About the same")
print(f"{'='*80}')")

Q5: RAG WITH CHUNKING

Query: How does the agentic loop keep calling the model until it stops?

Answer (from chunked index):
The agentic loop keeps calling the model until it stops by using a `while True` loop that continuously checks for function calls in each iteration. Here's how it works:

1. **Iteration Counter**: The loop increments an iteration counter (`it`) with each cycle, allowing tracking of how many times the loop has run.

2. **Response Handling**: In each iteration, the model is called and the response is processed to check for outputs. The model's response can contain various types of items, such as `f...

TOKEN COMPARISON

Q3 (Full documents - 72 docs):
  Input tokens: 7091

Q5 (Chunked index - 295 chunks):
  Input tokens: 2274

Difference:
  Tokens saved: 4817
  Percentage: 67.9% reduction

Ratio:
  ~3× fewer tokens
================================================================================')
